# Étape 2 — Séparer et préparer les images

**Termes techniques :** Train, validation, test, stratification, data leakage, batch, normalisation, tf.data


## Cellule 1 — Vérifier les outils et les chemins


In [1]:
from pathlib import Path
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_DIR = Path.cwd()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "data" / "raw" / "seg_train").is_dir():
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data" / "raw" / "seg_train"
SPLIT_DIR = PROJECT_DIR / "data" / "split"
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42

assert DATA_DIR.is_dir(), f"Dataset introuvable : {DATA_DIR}"
print("TensorFlow :", tf.__version__)
print("Dossier des classes :", DATA_DIR)
print("Classes trouvées :", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

TensorFlow : 2.21.0
Dossier des classes : c:\Users\Olaya\Desktop\sprint_2\SceneVision AI\data\raw\seg_train
Classes trouvées : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


## Cellule 2 — Construire une liste vérifiable des images


In [2]:
from PIL import Image

records = []
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
problem_files = []

for class_dir in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for image_path in sorted(class_dir.iterdir()):
        if image_path.is_file() and image_path.suffix.lower() in image_extensions:
            try:
                with Image.open(image_path) as image:
                    image.verify()
                records.append({"path": str(image_path.resolve()), "label": class_dir.name})
            except Exception as error:
                problem_files.append((str(image_path), str(error)))

df = pd.DataFrame(records)
class_names = sorted(df["label"].unique())
print("Images lisibles :", len(df))
print("Classes :", class_names)
print("Images problématiques :", len(problem_files))
display(df["label"].value_counts().sort_index().rename("nombre d'images").to_frame())
assert len(class_names) == 6, "Le projet attend six classes."
assert len(df) > 0, "Aucune image lisible détectée."

Images lisibles : 17034
Classes : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
Images problématiques : 0


,nombre d'images
label,
buildings,2628
forest,2745
glacier,2957
mountain,3037
sea,2784
street,2883


## Cellule 3 — Faire la séparation stratifiée 70/15/15


In [3]:
train_df, remainder_df = train_test_split(
    df, test_size=0.30, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    remainder_df, test_size=0.50, random_state=SEED, stratify=remainder_df["label"]
)

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
val_df.to_csv(SPLIT_DIR / "validation.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

summary = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
    "test": test_df["label"].value_counts(),
}).fillna(0).astype(int).sort_index()
display(summary)
print(f"Total : {len(train_df)} train + {len(val_df)} validation + {len(test_df)} test = {len(df)}")
print("Manifestes enregistrés dans :", SPLIT_DIR)
assert set(train_df.path).isdisjoint(val_df.path)
assert set(train_df.path).isdisjoint(test_df.path)
assert set(val_df.path).isdisjoint(test_df.path)

,train,validation,test
label,,,
buildings,1839,395,394
forest,1921,412,412
glacier,2070,443,444
mountain,2126,455,456
sea,1949,418,417
street,2018,432,433


Total : 11923 train + 2555 validation + 2556 test = 17034
Manifestes enregistrés dans : c:\Users\Olaya\Desktop\sprint_2\SceneVision AI\data\split


## Cellule 4 — Charger un lot et appliquer le prétraitement


In [4]:
import sys
sys.path.insert(0, str(PROJECT_DIR))
from src.data_pipeline import load_manifest

class_to_id = {name: index for index, name in enumerate(class_names)}
train_ds = load_manifest(SPLIT_DIR / "train.csv", class_to_id, IMG_SIZE, BATCH_SIZE, shuffle=True, seed=SEED)
val_ds = load_manifest(SPLIT_DIR / "validation.csv", class_to_id, IMG_SIZE, BATCH_SIZE)
test_ds = load_manifest(SPLIT_DIR / "test.csv", class_to_id, IMG_SIZE, BATCH_SIZE)

images, labels = next(iter(train_ds))
print("Forme du lot :", images.shape)
print("Forme des labels :", labels.shape)
print("Plage des pixels :", float(tf.reduce_min(images)), "à", float(tf.reduce_max(images)))
print("Ordre des classes :", class_to_id)

Forme du lot d'images : (32, 150, 150, 3)
Forme des labels : (32,)
Plage des pixels : 0.0 à 1.0
Ordre des classes : {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}


## Vérification et révision
